# 02_1 — Collapse editions → works (`work_id`)

Standalone fix for the duplicate-editions problem (UCSD Goodreads catalogs editions, not works). Post-processes the **existing** artifacts — no re-download, no GPU re-embed:

1. Build `book_id → work_id` from the raw books metadata.
2. `collapse_editions`: canonical (most-read) edition per work, remap interactions, dedupe per work, realign embeddings.
3. Save `*_works` artifacts.

Then point `07_models` at `sample_works.parquet` / `catalog_works.parquet` / `embeddings_works.npy`.

> Note: this collapses *after* the edition-level k-core/sample (a pragmatic shortcut). The fully-rigorous version collapses *before* k-core — see `TODO.md`.

In [ ]:
# Make the book_recsys package importable.
# Local: already installed (editable). Kaggle: find the repo among mounted inputs,
# else clone it from GitHub. (sys.path only — no pip install needed: this notebook
# uses just pandas/numpy from book_recsys.data.)
import sys, os, glob, subprocess
try:
    import book_recsys  # noqa: F401
except ModuleNotFoundError:
    hits = glob.glob('/kaggle/input/**/book_recsys/__init__.py', recursive=True)
    if hits:
        sys.path.insert(0, os.path.dirname(os.path.dirname(hits[0])))
    else:
        dst = '/kaggle/working/book_recsys_src'
        if not os.path.exists(dst):
            subprocess.run(['git', 'clone', '--depth', '1',
                            'https://github.com/MayaDeneva/book_recsys', dst], check=True)
        sys.path.insert(0, dst)
    import book_recsys  # noqa: F401
print('book_recsys at', book_recsys.__file__)

In [1]:
import glob
import numpy as np, pandas as pd
from book_recsys.data.books import stream_books_json
from book_recsys.data.works import collapse_editions

def find(name):
    # artifacts live in artifacts/ or ../artifacts/; raw books json may be in data/ or a
    # mounted Kaggle input dir — search broadly.
    pats = [name, f'artifacts/{name}', f'../artifacts/{name}', f'data/{name}',
            f'../data/{name}', f'/kaggle/input/**/{name}']
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits: return hits[0]
    raise FileNotFoundError(name)

In [2]:
sample = pd.read_parquet(find('sample.parquet'))
catalog = pd.read_parquet(find('catalog.parquet'))
emb = np.load(find('embeddings.npy'))
print(f'{len(catalog):,} books · {len(sample):,} interactions · emb {emb.shape}')

701,085 books · 15,708,425 interactions · emb (701085, 384)


In [3]:
catalog.head()

,book_id,title,description,language_code,shelves,author_id,author
0,7327624,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",Omnibus book club edition containing the Ladie...,eng,"[to-read, fantasy, fiction, owned, hardcover]",10333,Barbara Hambly
1,6066812,All's Fairy in Love and War (Avalon: Web of Ma...,"To Kara's astonishment, she discovers that a p...",,"[to-read, fantasy, owned, books-i-own, current...",19158,Rachel Roberts
2,6066814,"Crowner Royal (Crowner John Mystery, #13)","London, 1196. At the command of Richard the Li...",,"[to-read, historical-fiction, mystery, histori...",37778,Bernard Knight
3,33394837,The House of Memory (Pluto's Snitch #2),,eng,"[currently-reading, netgalley, kindle, read-20...",242185,Carolyn Haines
4,89377,Penny from Heaven,It's 1953 and 11-year-old Penny dreams of a su...,,"[to-read, historical-fiction, currently-readin...",137561,Jennifer L. Holm


### 1. Build `book_id → work_id` (stream the raw books metadata)

In [4]:
# locate the Goodreads books-metadata file (work_id lives here) under any common name
books_path = None
for _n in ['goodreads_books.json.gz', 'goodreads_books.json', 'books.json.gz', 'books.json']:
    try:
        books_path = find(_n); break
    except FileNotFoundError:
        continue
if books_path is None:
    raise FileNotFoundError('Goodreads books metadata (goodreads_books.json[.gz] / books.json)')
print('books metadata:', books_path)

catalog_ids = set(catalog['book_id'])
work_of = {}
for chunk in stream_books_json(books_path):
    m = chunk[chunk['book_id'].isin(catalog_ids)]
    work_of.update(zip(m['book_id'], m['work_id']))
work_of = {b: w for b, w in work_of.items() if w}   # drop blank work_ids (own work)
print(f'work_id for {len(work_of):,} / {len(catalog_ids):,} catalog books' + ('  <-- ~0 means this file has no work_id field!' if not work_of else ''))

books metadata: ../data/books.json.gz
work_id for 701,085 / 701,085 catalog books


### 2. Collapse

In [5]:
sample_w, catalog_w, emb_w = collapse_editions(sample, catalog, emb, work_of)
print(f'books {len(catalog):,} -> {len(catalog_w):,}  '
      f'({len(catalog) - len(catalog_w):,} editions merged)')
print(f'interactions {len(sample):,} -> {len(sample_w):,}')
print(f'embeddings {emb.shape} -> {emb_w.shape}')

books 701,085 -> 468,628  (232,457 editions merged)
interactions 15,708,425 -> 12,618,544
embeddings (701085, 384) -> (468628, 384)


In [7]:
catalog_w.head()

,book_id,title,description,language_code,shelves,author_id,author,work_id
0,7327624,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",Omnibus book club edition containing the Ladie...,eng,"[to-read, fantasy, fiction, owned, hardcover]",10333,Barbara Hambly,8948723
1,6066814,"Crowner Royal (Crowner John Mystery, #13)","London, 1196. At the command of Richard the Li...",,"[to-read, historical-fiction, mystery, histori...",37778,Bernard Knight,6243149
2,33394837,The House of Memory (Pluto's Snitch #2),,eng,"[currently-reading, netgalley, kindle, read-20...",242185,Carolyn Haines,54143148
3,89377,Penny from Heaven,It's 1953 and 11-year-old Penny dreams of a su...,,"[to-read, historical-fiction, currently-readin...",137561,Jennifer L. Holm,86258
4,89378,Dog Heaven,In Newbery Medalist Cynthia Rylant's classic b...,eng,"[to-read, picture-books, animals, children-s-b...",5411,Cynthia Rylant,86259


In [8]:
sample_w.head()

,user_id,book_id,rating,timestamp
0,d889b42d9eb7b80e02f24830e27c6389,196084,3,978336000
1,56771a556632bb4465cab31e5ab5ad81,285205,4,979027200
2,d889b42d9eb7b80e02f24830e27c6389,79030,3,979200000
3,d889b42d9eb7b80e02f24830e27c6389,442783,4,980409600
4,d889b42d9eb7b80e02f24830e27c6389,552719,2,980755200


### 3. Save work-level artifacts

In [ ]:
import os
art = os.path.dirname(find('sample_1.parquet'))
sample_w.to_parquet(f'{art}/sample.parquet')
catalog_w.to_parquet(f'{art}/catalog.parquet')
np.save(f'{art}/embeddings.npy', emb_w)
print('saved sample.parquet, catalog.parquet, embeddings.npy ->', art)

saved sample_works.parquet, catalog_works.parquet, embeddings_works.npy -> ../artifacts


### Use in `07_models`
Swap the loads to the `*_works` files:
```python
sample = pd.read_parquet(find('sample_works.parquet'))
catalog = pd.read_parquet(find('catalog_works.parquet'))
emb = np.load(find('embeddings_works.npy'))
```